In [0]:
# Databricks notebook source
# ===================================================================================
#
# JOB: BACKFILL PROCESSOR
#
# AUTHOR: GitHub Copilot & naveenbanappa
# VERSION: 1.2 (Definitive Correction)
#
# ===================================================================================

# COMMAND ----------

# MAGIC %md
# MAGIC ### Section 1: Imports & Job Parameters
# MAGIC *(This section is correct and remains unchanged)*

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

dbutils.widgets.text("target_table_name", "", "1. Full name of the table to backfill")
dbutils.widgets.text("old_schema_hash", "", "2. The 'bad' hash to find and fix")
dbutils.widgets.text("new_schema_hash", "", "3. The 'good' hash to apply after fixing")
dbutils.widgets.text("run_mode", "DRY_RUN", "4. Execution mode: DRY_RUN or WET_RUN")
dbutils.widgets.text("control_table_schema", "mq_gmdf_dp_poc.metadata_ddl", "5. UC Schema for Control Tables")


# COMMAND ----------

# MAGIC %md
# MAGIC ### Section 2: Configuration & Initialization

# COMMAND ----------

# --- Fetch parameters from widgets
target_table_name = dbutils.widgets.get("target_table_name")
old_schema_hash = dbutils.widgets.get("old_schema_hash")
new_schema_hash = dbutils.widgets.get("new_schema_hash")
run_mode = dbutils.widgets.get("run_mode").upper()
control_table_schema = dbutils.widgets.get("control_table_schema")

# --- Define control table names
schema_store_columns_table = f"{control_table_schema}.schema_store_columns"
schema_transition_table = f"{control_table_schema}.schema_transition"

# --- Basic validation
if not all([target_table_name, old_schema_hash, new_schema_hash, control_table_schema]):
    raise ValueError("FATAL: Parameters 'target_table_name', 'old_schema_hash', 'new_schema_hash', and 'control_table_schema' are all required.")

if run_mode not in ["DRY_RUN", "WET_RUN"]:
    raise ValueError(f"FATAL: Invalid run_mode '{run_mode}'. Must be 'DRY_RUN' or 'WET_RUN'.")

# --- THE ONLY REAL FIX NEEDED: The audit column in the *target data table*.
SCHEMA_HASH_COLUMN_NAME = "schema_hash_at_ingest"

# --- Log the configuration for this run
print("✅ Backfill Job Configuration Initialized")
print("=========================================")
print(f"   Target Table: {target_table_name}")
print(f"   Run Mode: {run_mode}")
print("-----------------------------------------")
print(f"   Old (Bad) Hash: {old_schema_hash}")
print(f"   New (Good) Hash: {new_schema_hash}")
print("=========================================")


# COMMAND ----------

# MAGIC %md
# MAGIC ### Section 3: Identify Schema Differences

# COMMAND ----------

print("🚀 Step 1: Identifying schema differences...")

try:
    # Fetch the full blueprint for the old and new schemas
    schema_store_df = spark.table(schema_store_columns_table).filter(F.col("target_table_name") == target_table_name)
    
    # --- REVERTED CHANGE: The column is indeed 'contract_hash' in this table.
    old_schema_cols = schema_store_df.filter(F.col("contract_hash") == old_schema_hash)
    new_schema_cols = schema_store_df.filter(F.col("contract_hash") == new_schema_hash)
    
    # Cache them for performance
    old_schema_cols.cache()
    new_schema_cols.cache()

    # --- REVERTED CHANGE: The column for the join is 'field_name_in_struct'.
    diff_df = new_schema_cols.join(
        old_schema_cols,
        new_schema_cols.field_name_in_struct == old_schema_cols.field_name_in_struct,
        "left_anti"
    )
    
    columns_to_backfill = diff_df.collect()
    
    if not columns_to_backfill:
        msg = "No new extractable columns found between the two schemas. Nothing to backfill. Marking as complete."
        print(f"✅ {msg}")
        spark.sql(f"UPDATE {schema_transition_table} SET backfill_status = 'COMPLETE' WHERE target_table_name = '{target_table_name}' AND schema_hash = '{new_schema_hash}'")
        dbutils.notebook.exit(msg)
        
    print(f"   - Found {len(columns_to_backfill)} new columns to backfill:")
    for row in columns_to_backfill:
        # --- REVERTED CHANGE: Using 'field_name_in_struct' and 'source_column'.
        print(f"     - Column: `{row['field_name_in_struct']}`, Source: `{row['source_column']}`")

except Exception as e:
    raise RuntimeError(f"FATAL: Could not compute schema differences from '{schema_store_columns_table}'. Error: {e}")


# COMMAND ----------

# MAGIC %md
# MAGIC ### Section 4: Find Target Rows & Generate UPDATE Logic
# MAGIC *(This section's logic was correct but now uses the right column names from Section 3)*

# COMMAND ----------

print("\n🚀 Step 2: Finding target rows and preparing UPDATE logic...")

# --- Find the rows in the target table that were written with the old, "bad" hash
try:
    rows_to_update_df = spark.table(target_table_name).filter(F.col(SCHEMA_HASH_COLUMN_NAME) == old_schema_hash)
    row_count = rows_to_update_df.count()
    
    if row_count == 0:
        msg = "No rows found with the old schema hash. Nothing to backfill. Marking as complete."
        print(f"✅ {msg}")
        spark.sql(f"UPDATE {schema_transition_table} SET backfill_status = 'COMPLETE' WHERE target_table_name = '{target_table_name}' AND schema_hash = '{new_schema_hash}'")
        dbutils.notebook.exit(msg)

    print(f"   - Found {row_count} rows in '{target_table_name}' that need backfilling.")

except AnalysisException as e:
    if "COLUMN_NOT_FOUND" in str(e) and SCHEMA_HASH_COLUMN_NAME in str(e):
        raise ValueError(f"FATAL: The target table '{target_table_name}' does not have the required audit column '{SCHEMA_HASH_COLUMN_NAME}'. Cannot proceed.")
    else:
        raise e

# --- Dynamically build the SET clauses for the SQL UPDATE statement
update_clauses = []
for row in columns_to_backfill:
    # --- REVERTED CHANGE: Using 'field_name_in_struct' and 'source_column'
    target_col_name = row['field_name_in_struct'] 
    source_json_col = row['source_column']
    source_json_path = f"$.{row['field_name_in_struct']}"
    
    update_clauses.append(f"`{target_col_name}` = get_json_object(`{source_json_col}`, '{source_json_path}')")

update_statement = f"""
    UPDATE {target_table_name}
    SET
        {', '.join(update_clauses)},
        `{SCHEMA_HASH_COLUMN_NAME}` = '{new_schema_hash}'
    WHERE
        `{SCHEMA_HASH_COLUMN_NAME}` = '{old_schema_hash}'
"""

print("   - Generated SQL UPDATE statement prepared.")


# COMMAND ----------

# MAGIC %md
# MAGIC ### Section 5: Execute Backfill (Dry or Wet Run)
# MAGIC *(This section is correct and remains unchanged)*

# COMMAND ----------

print(f"\n🚀 Step 3: Executing backfill in '{run_mode}' mode...")

if run_mode == "DRY_RUN":
    print("========================= DRY RUN REPORT =========================")
    print(f"This is a DRY RUN. No data will be changed.")
    print(f"Target Table: {target_table_name}")
    print(f"Action: UPDATE")
    print(f"Rows to be affected: {row_count}")
    print("\n--- Columns to be populated ---")
    for clause in update_clauses:
        print(f"  - {clause}")
    print("\n--- Final State ---")
    print(f"  - Rows' `{SCHEMA_HASH_COLUMN_NAME}` would be set to '{new_schema_hash}'.")
    print(f"  - Backfill status for hash '{new_schema_hash}' would be set to 'COMPLETE'.")
    print("\nTo apply these changes, run this job again with the parameter `run_mode` set to 'WET_RUN'.")
    print("==================================================================")
    dbutils.notebook.exit("Dry run complete. No changes were made.")

elif run_mode == "WET_RUN":
    try:
        print("   - Executing WET RUN. Data will be modified.")
        print("   - Running UPDATE statement on target table...")
        spark.sql(update_statement)
        print("   - ✅ UPDATE complete.")
        
        print("   - Updating schema_transition table to mark backfill as 'COMPLETE'...")
        spark.sql(f"UPDATE {schema_transition_table} SET backfill_status = 'COMPLETE' WHERE target_table_name = '{target_table_name}' AND schema_hash = '{new_schema_hash}'")
        print("   - ✅ Backfill status marked as COMPLETE.")
        
        final_msg = f"WET RUN successfully completed. {row_count} rows were backfilled and the process is now marked as complete."
        print(f"\n🎉 {final_msg}")
        dbutils.notebook.exit(final_msg)
        
    except Exception as e:
        error_msg = f"FATAL ERROR during WET_RUN execution: {e}"
        print(error_msg)
        try:
            spark.sql(f"UPDATE {schema_transition_table} SET backfill_status = 'FAIL' WHERE target_table_name = '{target_table_name}' AND schema_hash = '{new_schema_hash}'")
            print("   - Updated backfill status to 'FAIL' due to error.")
        except Exception as log_e:
            print(f"   - CRITICAL: Could not even update status to FAIL. Error: {log_e}")
        raise RuntimeError(error_msg)